# Unificação e Filtragem da Base GERCON — Município Executante (Porto Alegre - RS)

Este notebook realiza a leitura automatizada, junção de todos os datasets anuais (`.csv`) do sistema **GERCON** (2018 a 2026) presentes na pasta do Google Drive, aplica o filtro pelo **Município Executante: Porto Alegre** e padroniza a inclusão do **Código IBGE de Porto Alegre (`4314902` / `431490`)**.

---

### ⚙️ Estratégia de Processamento:
1. **Montagem do Google Drive**: Acesso direto aos arquivos da tese/doutorado.
2. **Leitura Eficiente**: Processamento individual arquivo a arquivo para manter o uso de memória RAM baixo no Colab.
3. **Detecção Flexível**: Tratamento automático de delimitador (`;` ou `,`) e codificação (`utf-8` ou `latin-1`).
4. **Filtragem por Município Executante**: Filtro por "PORTO ALEGRE" e/ou código IBGE na coluna correspondente (`MUNICIPIO_SOLICITANTE`).
5. **Inclusão do Código IBGE**: Criação e padronização explícita das colunas do IBGE (`4314902` de 7 dígitos e `431490` de 6 dígitos).
6. **Exportação**: Geração de um novo arquivo consolidado `.csv` gravado diretamente no seu Google Drive.


## 1. 🔗 Montagem do Google Drive


In [ ]:
from google.colab import drive
import os
import glob
import unicodedata
import gc
import pandas as pd
import numpy as np

# Montar o Google Drive
drive.mount('/content/drive')



Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. 📂 Localização dos Arquivos GERCON


In [ ]:
# Caminho da pasta no Google Drive: 01_Doutorado -> Tese -> banco -> gercon
caminho_gercon = '/content/drive/MyDrive/01_Doutorado/Tese/banco/gercon'

# Fallback automático caso o caminho varie no seu Drive
if not os.path.exists(caminho_gercon):
    candidatos = [
        '/content/drive/MyDrive/Doutorado/Tese/banco/gercon',
        '/content/drive/MyDrive/Tese/banco/gercon',
        '/content/drive/MyDrive/banco/gercon',
        '/content/drive/MyDrive/01_Doutorado/Tese/banco/gercom'
    ]
    for c in candidatos:
        if os.path.exists(c):
            caminho_gercon = c
            break

print(f"Diretório selecionado: {caminho_gercon}")

# Listar todos os arquivos CSV brutos (excluindo eventuais saídas consolidadas anteriores)
arquivos_csv = sorted([
    f for f in glob.glob(os.path.join(caminho_gercon, "*.csv"))
    if not os.path.basename(f).startswith("gercon_solicitacoes_executante_poa")
    and not os.path.basename(f).startswith("dados_poa")
])

print(f"\nTotal de arquivos CSV encontrados: {len(arquivos_csv)}")
for arq in arquivos_csv:
    tam_mb = os.path.getsize(arq) / (1024 * 1024)
    print(f"  • {os.path.basename(arq)} ({tam_mb:.1f} MB)")



Diretório selecionado: /content/drive/MyDrive/01_Doutorado/Tese/banco/gercon

Total de arquivos CSV encontrados: 10
  • gercon_solicitacoes_2018.csv (87.9 MB)
  • gercon_solicitacoes_2019.csv (168.1 MB)
  • gercon_solicitacoes_2020.csv (124.2 MB)
  • gercon_solicitacoes_2021.csv (169.6 MB)
  • gercon_solicitacoes_2022.csv (318.2 MB)
  • gercon_solicitacoes_2023.csv (391.0 MB)
  • gercon_solicitacoes_2024 (1).csv (370.4 MB)
  • gercon_solicitacoes_2025.csv (461.1 MB)
  • gercon_solicitacoes_2026.csv (135.9 MB)
  • gercon_solicitacoes_executante_porto_alegre_2018_2025.csv (876.2 MB)


In [ ]:
sample = pd.read_csv("/content/drive/MyDrive/01_Doutorado/Tese/banco/gercon/gercon_solicitacoes_2020.csv",dtype=str, sep = ";")


In [ ]:
sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1409118 entries, 0 to 1409117
Data columns (total 29 columns):
 #   Column                    Non-Null Count    Dtype 
---  ------                    --------------    ----- 
 0   data_extracao             1409118 non-null  object
 1   idsolicitacao             1409118 non-null  object
 2   situacaosolicitacao       1409118 non-null  object
 3   data_solicitacao          1409118 non-null  object
 4   corregulador              1409118 non-null  object
 5   diasfilaespera            1409118 non-null  object
 6   complexidade              1409118 non-null  object
 7   especialidade             1409118 non-null  object
 8   especialidademae          1409118 non-null  object
 9   cid                       1409118 non-null  object
 10  idpaciente                1409118 non-null  object
 11  municipio_residencia      1409118 non-null  object
 12  idade                     1409118 non-null  object
 13  cnes_solicitante          1409118 non-null

In [ ]:
# 1. Deixar a coluna 'municipiosolicitante' com todas as letras maiúsculas
sample['municipiosolicitante'] = sample['municipiosolicitante'].str.upper()

# 2. Contar a frequência relativa dos municípios que começam com 'P'
#municipios_com_p = sample[sample['municipiosolicitante'].str.startswith('P')]
frequencia_relativa_p = sample['municipiosolicitante'].value_counts(normalize=True)

print("Frequência relativa dos municípios que começam com 'P' na coluna 'municipiosolicitante':")
display(frequencia_relativa_p)

Frequência relativa dos municípios que começam com 'P' na coluna 'municipiosolicitante':


,proportion
municipiosolicitante,
PORTO ALEGRE,0.626203
ALVORADA,0.024090
VIAMAO,0.020956
GRAVATAI,0.020625
NOVO HAMBURGO,0.014695
...,...
NOVO XINGU,0.000003
SANTO ANTONIO DO PALMA,0.000003
SETE DE SETEMBRO,0.000003


## 3. 🛠️ Funções de Leitura, Padronização e Filtro

- **Código IBGE de Porto Alegre**: `4314902` (7 dígitos) e `431490` (6 dígitos).
- Tratamento para garantir a integridade dos tipos e caracteres com acentuação.


In [ ]:
# Constantes oficiais do IBGE para Porto Alegre / RS
IBGE_POA_7D = "4314902"
IBGE_POA_6D = "431490"
NOME_POA = "PORTO ALEGRE"

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8").strip().upper()

def carregar_e_filtrar_arquivo(caminho_arquivo):
    nome_base = os.path.basename(caminho_arquivo)
    print(f"\n➡️ Processando: {nome_base}...")

    df = None
    # Identificar separador e codificação
    for encoding in ["utf-8", "latin-1", "iso-8859-1"]:
        for sep in [";", ","]:
            try:
                sample = pd.read_csv(caminho_arquivo, sep=sep, encoding=encoding, nrows=10, dtype=str)
                if len(sample.columns) > 5:
                    df = pd.read_csv(caminho_arquivo, sep=sep, encoding=encoding, dtype=str, low_memory=False)
                    print(f"   ✓ Lido com sucesso (sep='{sep}', enc='{encoding}', total={len(df):,} linhas)")
                    break
            except Exception:
                continue
        if df is not None:
            break

    if df is None:
        raise ValueError(f"Não foi possível abrir o arquivo: {caminho_arquivo}")

    # Localizar a coluna do município executante
    colunas_map = {col.upper().strip(): col for col in df.columns}

    col_executante = None
    for cand in ["MUNICIPIOSOLICITANTE", "municipiosolicitante"]:
        if cand in colunas_map:
            col_executante = colunas_map[cand]
            break

    if col_executante is None:
        for c in df.columns:
            if "SOLICITANTE" in c.upper() and ("MUN" in c.upper() or "CIDADE" in c.upper()):
                col_executante = c
                break

    if col_executante is None:
        print(f"   ⚠️ Atenção: Coluna de município solicitante não encontrada em {nome_base}!")
        return pd.DataFrame()

    # Aplicar filtro por Porto Alegre
    serie_norm = df[col_executante].fillna("").apply(normalizar_texto)
    filtro_poa = (
        (serie_norm == NOME_POA) |
        (df[col_executante].str.strip() == IBGE_POA_7D) |
        (df[col_executante].str.strip() == IBGE_POA_6D) |
        (serie_norm.str.contains("PORTO ALEGRE"))
    )

    df_filtrado = df[filtro_poa].copy()

    # Inserir/Padronizar os códigos IBGE de Porto Alegre
    df_filtrado["COD_IBGE_MUNICIPIO_EXECUTANTE_7D"] = IBGE_POA_7D
    df_filtrado["COD_IBGE_MUNICIPIO_EXECUTANTE_6D"] = IBGE_POA_6D
    df_filtrado["UF_EXECUTANTE"] = "RS"
    df_filtrado["ARQUIVO_ORIGEM"] = nome_base

    print(f"   ✓ Linhas retidas (POA Executante): {len(df_filtrado):,} ({len(df_filtrado)/len(df)*100:.2f}% do arquivo)")

    del df
    gc.collect()
    return df_filtrado



## 4. 🔄 Processamento e Junção de Todos os Anos


In [ ]:
dfs_filtrados = []

for caminho in arquivos_csv:
    df_parcial = carregar_e_filtrar_arquivo(caminho)
    if not df_parcial.empty:
        dfs_filtrados.append(df_parcial)

if dfs_filtrados:
    df_gercon_poa_final = pd.concat(dfs_filtrados, ignore_index=True)
    print("\n" + "="*60)
    print("🎉 UNIFICAÇÃO CONCLUÍDA COM SUCESSO!")
    print(f"• Total consolidado de linhas (POA solicitante): {len(df_gercon_poa_final):,}")
    print(f"• Total de colunas: {df_gercon_poa_final.shape[1]}")
    print("="*60)
else:
    print("❌ Nenhum dado foi filtrado.")




➡️ Processando: gercon_solicitacoes_2018.csv...
   ✓ Lido com sucesso (sep=';', enc='utf-8', total=211,383 linhas)
   ✓ Linhas retidas (POA Executante): 137,885 (65.23% do arquivo)

➡️ Processando: gercon_solicitacoes_2019.csv...
   ✓ Lido com sucesso (sep=';', enc='utf-8', total=429,869 linhas)
   ✓ Linhas retidas (POA Executante): 256,417 (59.65% do arquivo)

➡️ Processando: gercon_solicitacoes_2020.csv...
   ✓ Lido com sucesso (sep=';', enc='utf-8', total=322,624 linhas)
   ✓ Linhas retidas (POA Executante): 202,028 (62.62% do arquivo)

➡️ Processando: gercon_solicitacoes_2021.csv...
   ✓ Lido com sucesso (sep=';', enc='utf-8', total=440,400 linhas)
   ✓ Linhas retidas (POA Executante): 276,659 (62.82% do arquivo)

➡️ Processando: gercon_solicitacoes_2022.csv...
   ✓ Lido com sucesso (sep=';', enc='utf-8', total=844,527 linhas)
   ✓ Linhas retidas (POA Executante): 319,446 (37.83% do arquivo)

➡️ Processando: gercon_solicitacoes_2023.csv...
   ✓ Lido com sucesso (sep=';', enc='utf-

## 5. 📊 Diagnóstico e Validação do Dataset Consolidado


In [ ]:
# Distribuição por ano/arquivo de origem
print("📌 Quantidade de solicitações por arquivo:")
print(df_gercon_poa_final["ARQUIVO_ORIGEM"].value_counts().sort_index())

# Visualizar primeiras linhas
print("\n📌 Amostra dos dados consolidados:")
display(df_gercon_poa_final.head(10))

# Checagem das colunas IBGE criadas
print("\n📌 Verificação dos Códigos IBGE de Porto Alegre:")
print(df_gercon_poa_final[["COD_IBGE_MUNICIPIO_EXECUTANTE_7D", "COD_IBGE_MUNICIPIO_EXECUTANTE_6D", "UF_EXECUTANTE"]].drop_duplicates())



📌 Quantidade de solicitações por arquivo:
ARQUIVO_ORIGEM
gercon_solicitacoes_2018.csv        137885
gercon_solicitacoes_2019.csv        256417
gercon_solicitacoes_2020.csv        202028
gercon_solicitacoes_2021.csv        276659
gercon_solicitacoes_2022.csv        319446
gercon_solicitacoes_2023.csv        289436
gercon_solicitacoes_2024 (1).csv    283098
gercon_solicitacoes_2025.csv        382931
Name: count, dtype: int64

📌 Amostra dos dados consolidados:


,data_extracao,idsolicitacao,situacaosolicitacao,data_solicitacao,corregulador,diasfilaespera,complexidade,especialidade,especialidademae,cid,...,centralregulacao,centralregulacaoorigem,tipoconsulta,coordenadoriasolicitante,regiaosaudesolicitante,possuiunidadeindicada,COD_IBGE_MUNICIPIO_EXECUTANTE_7D,COD_IBGE_MUNICIPIO_EXECUTANTE_6D,UF_EXECUTANTE,ARQUIVO_ORIGEM
0,2025-01-07 00:00:00.000,747495,REALIZADA,2018-02-20 00:00:00.000,AMARELO,17,MEDIA,CIRURGIA BUCOMAXILOFACIAL CEO,CIRURGIA BUCOMAXILOFACIAL,K011,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,SIM,4314902,431490,RS,gercon_solicitacoes_2018.csv
1,2025-01-07 00:00:00.000,756290,REALIZADA,2018-02-28 00:00:00.000,AZUL,14,MEDIA,CIRURGIA BUCOMAXILOFACIAL CEO,CIRURGIA BUCOMAXILOFACIAL,K021,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,SIM,4314902,431490,RS,gercon_solicitacoes_2018.csv
2,2025-01-07 00:00:00.000,806777,REALIZADA,2018-04-12 00:00:00.000,AZUL,1,MEDIA,CIRURGIA BUCOMAXILOFACIAL CEO,CIRURGIA BUCOMAXILOFACIAL,K011,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
3,2025-01-07 00:00:00.000,730573,REALIZADA,2018-02-01 00:00:00.000,LARANJA,5,MEDIA,PERIODONTIA,ODONTO PERIODONTIA,K061,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
4,2025-01-07 00:00:00.000,744023,ENCERRADA,2018-02-16 00:00:00.000,NENHUM,0,MEDIA,PERIODONTIA,ODONTO PERIODONTIA,K051,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
5,2025-01-07 00:00:00.000,742036,REALIZADA,2018-02-15 00:00:00.000,NENHUM,1,MEDIA,PERIODONTIA,ODONTO PERIODONTIA,K053,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
6,2025-01-07 00:00:00.000,741057,ENCERRADA,2018-02-14 00:00:00.000,NENHUM,2,MEDIA,PERIODONTIA,ODONTO PERIODONTIA,K029,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
7,2025-01-07 00:00:00.000,787302,REALIZADA,2018-03-27 00:00:00.000,AZUL,0,MEDIA,PERIODONTIA,ODONTO PERIODONTIA,K055,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
8,2025-01-07 00:00:00.000,819307,ENCERRADA,2018-04-23 00:00:00.000,AZUL,1,MEDIA,PERIODONTIA,ODONTO PERIODONTIA,S025,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv
9,2025-01-07 00:00:00.000,773576,REALIZADA,2018-03-14 00:00:00.000,LARANJA,19,MEDIA,INFECTOLOGIA HIV ADULTO,INFECTOLOGIA,Z21,...,REGULACAO AMBULATORIAL,REGULACAO AMBULATORIAL,PRIMEIRA,1ª CRS,REGIÃO DE SAÚDE 10,NÃO,4314902,431490,RS,gercon_solicitacoes_2018.csv



📌 Verificação dos Códigos IBGE de Porto Alegre:
  COD_IBGE_MUNICIPIO_EXECUTANTE_7D COD_IBGE_MUNICIPIO_EXECUTANTE_6D  \
0                          4314902                           431490   

  UF_EXECUTANTE  
0            RS  


## 6. 💾 Exportação para Arquivo .CSV Consolidado no Google Drive


In [ ]:
# Nome do arquivo de saída consolidado
nome_arquivo_saida = "gercon_solicitacoes_executante_porto_alegre_2018_2025.csv"
caminho_saida = os.path.join(caminho_gercon, nome_arquivo_saida)

print(f"Salvando dataset consolidado em:\n{caminho_saida} ...")

# Exportar como CSV separado por ponto-e-vírgula e codificação UTF-8 com BOM (compatível com R, Python e Excel)
df_gercon_poa_final.to_csv(caminho_saida, sep=";", index=False, encoding="utf-8-sig")

tamanho_mb = os.path.getsize(caminho_saida) / (1024 * 1024)
print(f"\n✅ Arquivo gerado com sucesso!")
print(f"📦 Tamanho do arquivo: {tamanho_mb:.2f} MB")
print(f"📁 Caminho no Drive: {caminho_saida}")



Salvando dataset consolidado em:
/content/drive/MyDrive/01_Doutorado/Tese/banco/gercon/gercon_solicitacoes_executante_porto_alegre_2018_2025.csv ...

✅ Arquivo gerado com sucesso!
📦 Tamanho do arquivo: 876.24 MB
📁 Caminho no Drive: /content/drive/MyDrive/01_Doutorado/Tese/banco/gercon/gercon_solicitacoes_executante_porto_alegre_2018_2025.csv
